# L2d: Building and Testing a Defensive Fibonacci Program

You already met the Fibonacci recurrence in Monday's lecture. Today we turn that calculation into an interface someone else could call without reading its source: one that states how it is indexed, what it returns, and where it stops.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Turn a recurrence into a defensible interface:__ Convert the Fibonacci recurrence into an iterative function whose indexing convention and return representation are stated explicitly rather than left for the caller to infer.
> * __Treat base cases and invalid input as ordinary cases:__ Handle the base cases with an early return, and reject indices the interface does not support, rather than assuming callers will stay inside the intended range.
> * __Recognize a type limit as an API boundary:__ Explain why an `Int64` result forces this interface to stop at a specific index, and why stopping deliberately beats overflowing silently.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/). `Include.jl` also pulls in [`src/Compute.jl`](src/Compute.jl), which is where the function you are about to write lives.

___

## The interface, not the recurrence

The recurrence is not the interesting part today: $F_0=0$, $F_1=1$, and $F_n=F_{n-1}+F_{n-2}$ for $n\ge2$. You have seen it, and Week 3 will return to it to compare iteration against recursion.

What we have not settled is everything a caller actually needs to know.

> __The decisions that make it an interface:__
>
> * __What comes back?__ The whole vector `[F₀, F₁, …, Fₙ]`, not just $F_n$. A caller who wants only the last value can take it; a caller who wants the sequence would otherwise have to call us $n$ times.
> * __How is it indexed?__ Julia arrays start at `1` and the mathematics starts at $F_0$, so position `n + 1` holds $F_n$. That offset has to be stated, because it cannot be guessed.
> * __Where does it stop?__ $F_{92}$ is the largest Fibonacci number that fits in an `Int64`. The interface stops there on purpose.

___

## Implement the function

[The `fibonacci_sequence(...)` function](src/Compute.jl) lives in [`src/Compute.jl`](src/Compute.jl), which currently holds a signature, a docstring, and four `TODO` comments. Filling them in is the work of this lab.

> __The algorithm:__
>
> 1. Reject input outside the contract: a `Bool`, a negative index, or an index above 92.
> 2. Allocate `n + 1` integer positions.
> 3. Store $F_0$, and return early when `n == 0`. At `n == 0` the vector has exactly one slot, so writing $F_1$ into position 2 would raise [a `BoundsError`](https://docs.julialang.org/en/v1/base/base/#Core.BoundsError).
> 4. Store $F_1$, then fill each later position from the two before it.
> 5. Return the vector.

Two details worth knowing before you start:

> __Julia specifics:__
>
> * `Bool` is a subtype of `Integer`, so `fibonacci_sequence(true)` reaches the typed method. `true` is not a sequence index, so reject it explicitly with `n isa Bool`.
> * [`Base.Checked.checked_add`](https://docs.julialang.org/en/v1/base/math/#Base.Checked.checked_add) adds two integers and raises on overflow instead of wrapping around silently. Bounding `n` at 92 already prevents overflow, so this is a second line of defence: cheap, and it documents the hazard.

Open the file, complete all four `TODO`s, then restart the kernel and run from the top. The test cell near the end is your specification. Until then the next cell stops with a "not implemented yet" error, which is the expected starting state.

Let's compute the first ten numbers, storing the result in the `sequence::Vector{Int64}` variable:

In [ ]:
sequence = fibonacci_sequence(10)
(sequence = sequence, F10 = sequence[10 + 1])

___

## Base cases are ordinary supported inputs

`n = 0` and `n = 1` are not special pleading; they are the smallest legitimate requests this interface accepts. The early return in step 3 is what stops the loop writing $F_1$ into a vector that was only supposed to reach $F_0$.

Both should come back as complete, correctly-sized vectors:

In [ ]:
base_cases = (n0 = fibonacci_sequence(0), n1 = fibonacci_sequence(1))

___

## Representation changes the access pattern

We returned a vector. A dictionary keyed on the mathematical index is the obvious alternative, and it removes the `n + 1` offset entirely, at the cost of hashing every lookup and storing every key.

> __Which is right?__ For a sequence built in order and read in order, the vector wins: contiguous storage, no hashing, and the offset is a documented constant rather than a per-access cost. A dictionary earns its overhead when the keys are sparse or unordered, which is exactly what this sequence is not. This is the collections trade-off from Wednesday, in a concrete case.

Both views hold the same numbers; only the access pattern differs:

In [ ]:
dictionary_view = Dict((position - 1) => value for (position, value) in enumerate(sequence))
(vector_F7 = sequence[8], dictionary_F7 = dictionary_view[7])

___

## Where the interface stops

An `Int64` holds values up to $2^{63}-1$. $F_{92}$ fits; $F_{93}$ does not. Written with ordinary `+` and no bound, this function would return a negative number for $F_{93}$ and tell no one. Ours uses [the `Base.Checked.checked_add(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.Checked.checked_add), so it would raise [an `OverflowError`](https://docs.julialang.org/en/v1/base/base/#Core.OverflowError) instead; the bound at 92 is what turns that late, low-level error into [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) at the interface, where the caller can actually read it.

> __A type limit is a real boundary:__ Stopping at 92 is not an arbitrary restriction. It is the honest edge of what this return type can represent, and stating it is more useful than silently wrapping around. A caller who needs $F_{93}$ needs a different return type, and should be told so rather than handed a wrong number.

Let's confirm both sides of that boundary:

In [ ]:
largest_supported = last(fibonacci_sequence(92))
overflow_message = try
    fibonacci_sequence(93)
    "no error"
catch error
    sprint(showerror, error)
end
(F92 = largest_supported, F93 = overflow_message)

___

## Test the complete contract

These tests are the specification. They cover the base cases, an ordinary case, the representation comparison, both edges of the supported range, and each kind of input the interface refuses.

Do they all pass?

In [ ]:
@testset "defensive Fibonacci program" begin
    @test base_cases.n0 == [0]
    @test base_cases.n1 == [0, 1]
    @test sequence == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
    @test dictionary_view[10] == 55
    @test largest_supported == 7_540_113_804_746_346_429
    @test_throws ArgumentError fibonacci_sequence(-1)
    @test_throws ArgumentError fibonacci_sequence(2.5)
    @test_throws ArgumentError fibonacci_sequence(93)
    @test_throws ArgumentError fibonacci_sequence(true) # Bool <: Integer, but not an index
end

___

## Summary
Turning a recurrence into an interface means deciding what it returns, how it is indexed, and where it stops, and then defending those decisions in code.

> __Key Takeaways:__
>
> * **Indexing and representation are interface decisions:** Returning the whole vector, and placing each Fibonacci number one position later than its mathematical index, are choices a caller cannot guess, so they belong in the docstring and in the tests.
> * **Base cases and invalid input deserve tests:** The smallest legitimate inputs and the inputs you refuse are where interfaces break, and both are cheaper to test than to debug later.
> * **A type limit can define an API boundary:** Stopping at the largest index the return type can represent is more honest than overflowing, and it tells a caller who needs more that they need a different type.

Week 3 returns to this same calculation to compare iteration against recursion. The implementation you just wrote is the baseline it will be measured against.
___